# Data Exploration - Reanalsysis (ERA5 and MERRA2)

In [1]:
import pathlib
import datetime

In [2]:
import numpy

In [3]:
import xarray

In [4]:
import matplotlib.pyplot
import cartopy

In [5]:
import site_archive_jasmin

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/nopw/j04/mohc_shared/dscop/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0Radar', 'ew4_imerg_precip': '/gws/nopw/j04/ew4energy/imerg_2025_summer', 'ew4_merra2_meteo': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_meteo', 'ew4_merra2_aero': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_aerosols/3d', 'ew4_mtg_li': '/gws/nopw/j04/ew4energy/MTG_LI/', 'ew4_era5': '/gws/nopw/j04/ew4energy/ERA5/tutorial_202606'}


In [6]:
import pyearthtools.data
import pyearthtools.pipeline


In [7]:
import torch

### Define common parameters

In [8]:
select_dt = datetime.datetime(2025,5,11,15,0)

In [ ]:
ghana_extents = {
    'latitude': (4.5,12.3),
    'longitude': (-3.8, 4.0),
}

In [ ]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][0],
                 ghana_extents['longitude'][1],
                )

# ERA5 Reanalsyis Dataset

### Load in PyEarthTools

As with the other datasets, we can create a dataaet accessor to abstract away the details of accessing the data.

In [ ]:
ew4_era5_accessor = pyearthtools.data.archive.ew4_era5(variables=['temperature','specific_humidity', 'vertical_velocity'])

In [ ]:
ew4_era5_accessor[select_dt]

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,12))
ax1=fig1.add_subplot(1,1,1,projection=cartopy.crs.PlateCarree())
ew4_era5_accessor[select_dt].sel(pressure_level=500)['w'].squeeze().plot.contourf(ax=ax1)
ax1.coastlines(color='w')



## Train an ML model
To demonstrate how we can use this data in a machine learning training pipeline, we will show a simple autoencoder

In [ ]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
ew4_era5_prep = pyearthtools.pipeline.Pipeline(
    ew4_era5_accessor,
    
    pyearthtools.data.transform.region.Bounding(*ghana_pet_box),
    exceptions_to_ignore=pyearthtools.data.exceptions.DataNotFoundError,
)

In [ ]:
ew4_era5_prep[select_dt]

In [ ]:
ew4_era5_ml = pyearthtools.pipeline.Pipeline(
    pyearthtools.pipeline.operations.xarray.reshape.CoordinateFlatten(['pressure_level']),
    pyearthtools.pipeline.operations.xarray.conversion.ToNumpy(),
    pyearthtools.pipeline.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
)


We can build up our total pipeline from these small pipeline components

In [ ]:
(ew4_era5_prep | ew4_era5_ml)[select_dt]

In [ ]:
train_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('1 hour')),
    iterator=petpipe.iterators.DateRange('20250501T00', '20250701T00', interval='1 hour').randomise(), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)
val_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('1 hour')),
    iterator=petpipe.iterators.DateRange('20250701T00', '20250801T00', interval='1 hour'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

In [ ]:
ew4_era5_train_pipe = ew4_imerg_prep | ew4_imerg_ml | train_range
ew4_era5_val_pipe = ew4_imerg_prep | ew4_imerg_ml | val_range

In [ ]:
numpy.histogram(ew4_era5_val_pipe['2025-07-28 18:00'])


## Set up autoencoder architecture

In [ ]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

### Determine the size of the filters around the latent space
One data dpeendant thing is to determine the size of the convolutional blocks either side of the latent space. We create some dummy layer to apply to our data to determine the size.

In [ ]:
sample_tensor = torch.tensor(next(iter(ew4_era5_train_pipe))[0][0], dtype=torch.float32).to(device)

In [ ]:
sample_tensor

In [ ]:
sample_tensor.shape

In [ ]:
torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, 
                            out_channels=16, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=16, 
                            out_channels=32, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
).to(device)(sample_tensor).shape

### Create model class

In [ ]:
class ERA5AutoEncoder(torch.nn.Module):
    def __init__(self, input_channels, max_pool=False):
        super(ImergAutoEncoder, self).__init__()

        # we have "hard coded" a lot of the architecture hyperparameters in our model class. 
        # Usually you want want to make these arguments for the class so you can vary hyperparameters more easily.
        # Hard coding here makes it easier to follow the architecture definition in the tutorial
        
        self._num_channels = [input_channels, 16,32]
        self._latent_array_dims = (-1,self._num_channels[-1],8,8)
        self._prelatent_size = functools.reduce(lambda a,b:a*b, self._latent_array_dims[1:])
        # self._latent_size = 500
        
        self._encoder = self._get_encoder(max_pool)
        self._decoder = self._get_decoder()

    def _get_encoder(self, max_pool):

        encoder = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=self._num_channels[0], 
                            out_channels=self._num_channels[1], 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=self._num_channels[1], 
                            out_channels=self._num_channels[2], 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Flatten(1, -1),
            )
        return encoder

    def _get_decoder(self):
        """
        """
        decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(in_channels=self._num_channels[2], out_channels=self._num_channels[1], kernel_size=2,stride=2),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(in_channels=self._num_channels[1], out_channels=self._num_channels[0], kernel_size=2,stride=2),
        )
        return decoder
        
    def forward(self, x):

        # Get latent representation
        latent = self._encoder(x)

        # Reconstruct input
        reconstructed = self._decoder(latent.view(self._latent_array_dims))

        return reconstructed

In [ ]:
num_channels = sample_tensor.shape[1]
num_channels

In [ ]:
# Initialize model and move to device
era5_autoencoder = ERA5AutoEncoder(num_channels, False).to(device)

It is useful at this point to ensure that we have correctly structured and dimensioned our layers by doing a forward pass on our data. If there is a mismatch between layers, we will find this error before we try to start training.

In [ ]:
era5_autoencoder._encoder(sample_tensor).view((-1,32,16,16))

In [ ]:
sample_tensor.shape,era5_autoencoder.forward(sample_tensor).shape

### Set up and run the training loop
Now we set up the loop to run gradient descent with back propogation to optimise the model weights of the autoencoder

In [ ]:
# Loss function and optimizer
# loss_function = torch.nn.L1Loss()
# criterion = nn.KLDivLoss()
loss_function = torch.nn.MSELoss()

optimizer = torch.optim.Adam(imerg_autoencoder.parameters(), 
                             lr=5e-3)

In [ ]:
num_epochs = 25

In [ ]:
num_samples = len(ew4_era5_train_pipe)
batch_size=8
num_batches = math.ceil(num_samples / batch_size)
num_batches

In [ ]:
def get_batch_tensors(batch_size, ds_iterator, device):
    """
    Get a torch tensor for target and predictors from a pyearthtools pipeline iterator
    """
    predictor_batch_list = []
    target_batch_list = []
    for _ in range(batch_size):
        pred_sample, target_sample = next(ds_iterator)
        predictor_batch_list += [pred_sample[0]]
        target_batch_list += [target_sample[0]]
    
    # create numpy array of a batch 
    predictor_array = numpy.concat(predictor_batch_list, axis=0)
    target_array = numpy.concat(target_batch_list, axis=0)
        
    # convert to a tensor and send to gpu
    predictor_gpu_tensor = torch.tensor(
        predictor_array,
        dtype=torch.float32,
    ).to(device)
    target_gpu_tensor = torch.tensor(
        target_array,
        dtype=torch.float32,
    ).to(device)    
    
    return predictor_gpu_tensor, target_gpu_tensor
        
    

To be sure of our model set up, run a forward pass with a batch to check it works as it did with a single sample

In [ ]:
test_it = iter(ew4_era5_train_pipe)
imerg_autoencoder.forward(get_batch_tensors(batch_size, test_it, device)[0]).shape, imerg_autoencoder.forward(get_batch_tensors(batch_size, test_it, device)[0]).shape

In [ ]:
%%time
for epoch_num in range(num_epochs):
    print(epoch_num)
    epoch_train_loss = 0.0
    epoch_val_loss = 0.0
    
    era5_train_iter = iter(ew4_era5_train_pipe)
    
    for batch_ix in range(num_batches):

        predictor_gpu_tensor, target_gpu_tensor = get_batch_tensors(batch_size, imerg_train_iter, device)
        
        if (batch_ix % 100) == 0:
            print(batch_ix)

        # do training for batch
        optimizer.zero_grad()
        predictions = era5_autoencoder.forward(predictor_gpu_tensor)
        loss_batch = loss_function(predictions, target_gpu_tensor)
        loss_batch.backward()
        optimizer.step()
        epoch_train_loss += loss_batch.to('cpu').item()
    epoch_train_loss /= num_batches

    # calculate loss on validation data    
    for val_predictor, val_target in ew4_era5_val_pipe:
        val_pred_tensor = torch.tensor(val_predictor[0], dtype=torch.float32).to(device)
        predictions_val = imerg_autoencoder.forward( val_pred_tensor)
        val_target_tensor = torch.tensor(val_target[0], dtype=torch.float32).to(device)
        loss_batch_val = loss_function(predictions_val, val_target_tensor)
        epoch_val_loss += loss_batch_val.to('cpu').item()
    
    epoch_val_loss /= len(ew4_era5_val_pipe)
    
    print(epoch_train_loss)
    print(epoch_val_loss)
        
        
        
        
